# 04 — TF-IDF Analysis

Investigates how term richness and vocabulary specificity vary across popularity deciles.
All heavy computation is sampled and cached in `metadata.json`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent.parent))
from notebooks.eval.shared_setup import *

## TF-IDF Distribution Per Decile (Corpus-Wide)

In [ ]:
from src.metrics.tfidf_service import (
    load_or_compute_tfidf_stats,
    print_tfidf_stats,
    plot_tfidf_stats,
)
from src.metrics.decile_utils import boundaries_for

TFIDF_SAMPLE_PER_DECILE = 20_000
TFIDF_MAX_FEATURES      = 50_000

boundaries = boundaries_for(DECILE_MODE, boundaries_uw, boundaries_cw)

tfidf_stats = load_or_compute_tfidf_stats(
    metadata_path=metadata_path,
    corpus_path=CORPUS_PATH,
    boundaries=boundaries,
    sample_per_decile=TFIDF_SAMPLE_PER_DECILE,
    max_features=TFIDF_MAX_FEATURES,
    chunk_size=1000,
    chunk_overlap=100,
)

print_tfidf_stats(tfidf_stats)

plot_tfidf_stats(
    tfidf_stats,
    decile_mode=DECILE_MODE,
    sample_per_decile=TFIDF_SAMPLE_PER_DECILE,
    max_features=TFIDF_MAX_FEATURES,
    out_path=RESULTS_DIR / 'tfidf_distribution_per_decile.png',
    chunk_overlap=100,
    chunk_size=1000,
)
print('✓ Saved → tfidf_distribution_per_decile.png')

## TF-IDF Per Decile — Per Dataset Breakdown

In [ ]:
from src.metrics.tfidf_service import (
    load_or_compute_corpus_vectorizer,
    compute_tfidf_stats_for_ids,
)

CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 100
MAX_FEATURES  = 50_000
SAMPLE        = 20_000

print('Loading corpus-wide TF-IDF vectorizer…')
corpus_vec = load_or_compute_corpus_vectorizer(
    metadata_path=metadata_path,
    corpus_path=CORPUS_PATH,
    boundaries=boundaries,
    sample_per_decile=SAMPLE,
    max_features=MAX_FEATURES,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

PANEL_CFG = [
    ('mean_tf_score',          'Mean Sublinear TF',               'A. Mean TF per Chunk',            '#4e79a7'),
    ('mean_idf_of_used_terms', 'Mean IDF of Used Terms (corpus)', 'B. Mean IDF (corpus-wide terms)', '#f28e2b'),
    ('mean_chunk_length',      'Avg Chunk Length (chars)',         'C. Avg Chunk Length',             '#59a14f'),
    ('mean_unique_terms',      'Mean Unique Terms / Chunk',        'D. Vocabulary Breadth',           '#e15759'),
]
SE_MAP = {
    'mean_tf_score':          'se_tf_score',
    'mean_idf_of_used_terms': 'se_idf_of_used_terms',
    'mean_chunk_length':      'se_chunk_length',
    'mean_unique_terms':      'se_unique_terms',
}
d_idx = list(range(1, 11))

for strategy in ALL_STRATEGIES:
    df = results_by_strategy[strategy]
    if not GROUP_COL or GROUP_COL not in df.columns:
        print(f'⚠ No group column for {strategy_label(strategy)} — skipping per-dataset breakdown')
        continue

    per_group_stats = compute_tfidf_stats_for_ids(
        query_df=df[['wikipedia_id', decile_col, GROUP_COL]],
        corpus_path=CORPUS_PATH,
        decile_col=decile_col,
        group_col=GROUP_COL,
        corpus_vectorizer=corpus_vec,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )

    datasets = sorted(per_group_stats.keys())
    if not datasets:
        continue

    fig, axes = plt.subplots(len(datasets), 4,
                             figsize=(24, 4 * len(datasets)), squeeze=False)

    for row_idx, ds in enumerate(datasets):
        ds_stats = per_group_stats[ds]
        sub = df[df[GROUP_COL].astype(str) == ds]
        q_counts = [int((sub[decile_col] == d).sum()) for d in range(10)]

        for col_idx, (stat_key, ylabel, panel_label, color) in enumerate(PANEL_CFG):
            ax = axes[row_idx, col_idx]
            vals = [ds_stats[d].get(stat_key, float('nan')) if d in ds_stats else float('nan')
                    for d in range(10)]
            err_key = SE_MAP.get(stat_key)
            errs = ([ds_stats[d].get(err_key, 0.0) if d in ds_stats else 0.0 for d in range(10)]
                    if err_key else None)

            ax.bar(d_idx, vals, color=color, alpha=0.80, edgecolor='white', linewidth=1)
            if errs:
                ax.errorbar(d_idx, vals, yerr=errs, fmt='none', color='black',
                            capsize=3, linewidth=1.2, elinewidth=0.9)
            y_max = max((v for v in vals if v == v), default=1.0)
            for xi, v, qc in zip(d_idx, vals, q_counts):
                if qc > 0:
                    ax.text(xi, (v if v == v else 0.0) + y_max * 0.015,
                            f'n={qc}', ha='center', va='bottom', fontsize=6, color='#333')
            ax.set_xlabel('Decile (1=Rare → 10=Famous)', fontsize=8, fontweight='bold')
            ax.set_ylabel(ylabel, fontsize=8, fontweight='bold')
            ax.set_title(f'{panel_label}\n{ds}', fontsize=8, fontweight='bold')
            ax.set_xticks(d_idx)
            ax.grid(axis='y', alpha=0.25)

    fig.suptitle(
        f'TF-IDF Stats of Target Documents per Decile by Dataset\n'
        f'({strategy_label(strategy)}, mode={DECILE_MODE}, '
        f'chunk {CHUNK_SIZE} chars/{CHUNK_OVERLAP} overlap)',
        fontsize=12, fontweight='bold', y=1.01,
    )
    plt.tight_layout()
    out = RESULTS_DIR / f'tfidf_per_dataset_{strategy}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Saved → {out.name}')

## TF-IDF Trend — % Change vs Grand-Mean Baseline

In [ ]:
TREND_PANELS = [
    ('mean_tf_score',          'se_tf_score',          '% change vs grand mean', 'A. Mean Sublinear TF',           '#4e79a7'),
    ('mean_idf_of_used_terms', 'se_idf_of_used_terms', '% change vs grand mean', 'B. Mean IDF (corpus-wide)',      '#f28e2b'),
    ('mean_chunk_length',      'se_chunk_length',      '% change vs grand mean', 'C. Avg Chunk Length',            '#59a14f'),
    ('mean_unique_terms',      'se_unique_terms',      '% change vs grand mean', 'D. Vocab Breadth (unique terms)', '#e15759'),
]

for strategy in ALL_STRATEGIES:
    df = results_by_strategy[strategy]
    if not GROUP_COL or GROUP_COL not in df.columns:
        continue

    pgs = compute_tfidf_stats_for_ids(
        query_df=df[['wikipedia_id', decile_col, GROUP_COL]],
        corpus_path=CORPUS_PATH,
        decile_col=decile_col,
        group_col=GROUP_COL,
        corpus_vectorizer=corpus_vec,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )

    datasets = sorted(pgs.keys())
    if not datasets:
        continue

    fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
    cmap = plt.get_cmap('tab10')

    for col_idx, (stat_key, se_key, ylabel, panel_label, _color) in enumerate(TREND_PANELS):
        ax = axes[col_idx]
        all_pct = []
        for ds_idx, ds in enumerate(datasets):
            ds_stats = pgs[ds]
            color = cmap(ds_idx % 10)
            vals = np.array([ds_stats[d].get(stat_key, np.nan) if d in ds_stats else np.nan
                             for d in range(10)], dtype=float)
            errs = np.array([ds_stats[d].get(se_key, 0.0) if d in ds_stats else 0.0
                             for d in range(10)], dtype=float)
            finite = np.isfinite(vals)
            if not finite.any():
                continue
            baseline = float(np.nanmean(vals))
            if baseline == 0.0:
                continue
            se_mean = float(np.sqrt(np.nansum(errs ** 2))) / float(finite.sum())
            pct = (vals - baseline) / abs(baseline) * 100.0
            pct_err = np.sqrt(errs ** 2 + se_mean ** 2) / abs(baseline) * 100.0
            all_pct.extend(pct[np.isfinite(pct)].tolist())
            ax.plot(d_idx, pct, marker='o', markersize=4, linewidth=1.6, color=color, label=ds)
            ax.fill_between(d_idx, pct - pct_err, pct + pct_err, alpha=0.15, color=color)

        ax.axhline(0.0, color='grey', linewidth=1.0, linestyle='--', alpha=0.7)
        if all_pct:
            margin = max(abs(np.nanmax(all_pct)), abs(np.nanmin(all_pct))) * 1.25
            ax.set_ylim(-max(margin, 5.0), max(margin, 5.0))
        ax.set_xlabel('Decile (1 = Rare → 10 = Famous)', fontsize=8, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(panel_label, fontsize=9, fontweight='bold')
        ax.set_xticks(d_idx)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:+.0f}%'))
        ax.grid(alpha=0.25)
        if col_idx == 0:
            ax.legend(fontsize=6, loc='best', framealpha=0.7)

    fig.suptitle(
        f'TF-IDF Metric Trends — % Change vs Grand Mean\n'
        f'({strategy_label(strategy)}, mode={DECILE_MODE})'
        '  |  Shaded = ±1 SE  |  Dashed = 0%',
        fontsize=10, fontweight='bold',
    )
    plt.tight_layout()
    out = RESULTS_DIR / f'tfidf_trend_pct_{strategy}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Saved → {out.name}')

## Retrieval Difficulty Per Decile

In [ ]:
from src.metrics.tfidf_service import plot_retrieval_difficulty

results_df = results_by_strategy[ALL_STRATEGIES[0]]

fig = plot_retrieval_difficulty(
    corpus_docs=corpus_docs,
    corpus_chunks=corpus_chunks,
    questions_per_decile=np.array(
        [results_df[decile_col].value_counts().get(d, 0) for d in range(10)], dtype=float
    ),
    tfidf_stats=tfidf_stats,
    decile_mode=DECILE_MODE,
    out_path=RESULTS_DIR / 'retrieval_difficulty_per_decile.png',
)
plt.show()
print('✓ Saved → retrieval_difficulty_per_decile.png')